# =====================================================
# NOTEBOOK INFORMATION
# =====================================================

# Privacy-Preserving Federated Network Intrusion Detection System

## Notebook 04 - Feature Engineering

### Objectives

- Load the clean CICIDS2017 dataset
- Split data into training and testing sets
- Remove constant features using training data
- Identify highly correlated features using training data
- Apply the same feature selection to test data
- Save feature selection information

### Important

Feature selection is performed only on the training data
to prevent information leakage from the test set.

In [1]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# =====================================================
# PROJECT CONFIGURATION
# =====================================================

PROJECT_ROOT = Path.cwd().parent

CLEAN_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "CICIDS2017_clean.csv"
)

PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

RESULTS_PATH = PROJECT_ROOT / "results"
METRICS_PATH = RESULTS_PATH / "metrics"

METRICS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Clean dataset: {CLEAN_DATA_PATH}")

Clean dataset: c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\processed\CICIDS2017_clean.csv


In [3]:
# =====================================================
# LOAD CLEAN DATASET
# =====================================================

df = pd.read_csv(
    CLEAN_DATA_PATH
)

print(f"Dataset shape: {df.shape}")

df.head()

Dataset shape: (2520798, 79)


,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,...,min_seg_size_forward,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
# =====================================================
# SEPARATE FEATURES AND TARGET
# =====================================================

X = df.drop(
    columns=["Label"]
)

y = df["Label"]

print(f"Features: {X.shape}")
print(f"Target  : {y.shape}")

print("\nNumber of classes:", y.nunique())

Features: (2520798, 78)
Target  : (2520798,)

Number of classes: 15


In [5]:
# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples : {len(X_test):,}")

Training samples: 2,016,638
Testing samples : 504,160


In [6]:
# =====================================================
# IDENTIFY CONSTANT FEATURES
# =====================================================

constant_features = [
    column
    for column in X_train.columns
    if X_train[column].nunique() <= 1
]

print(
    f"Constant features found: "
    f"{len(constant_features)}"
)

if constant_features:
    print(constant_features)

Constant features found: 8
['Bwd_PSH_Flags', 'Bwd_URG_Flags', 'Fwd_Avg_Bytes_Bulk', 'Fwd_Avg_Packets_Bulk', 'Fwd_Avg_Bulk_Rate', 'Bwd_Avg_Bytes_Bulk', 'Bwd_Avg_Packets_Bulk', 'Bwd_Avg_Bulk_Rate']


In [7]:
# =====================================================
# REMOVE CONSTANT FEATURES
# =====================================================

X_train = X_train.drop(
    columns=constant_features
)

X_test = X_test.drop(
    columns=constant_features
)

print(
    f"Training features after removal: "
    f"{X_train.shape[1]}"
)

Training features after removal: 70


In [8]:
# =====================================================
# IDENTIFY HIGHLY CORRELATED FEATURES
# =====================================================

correlation_matrix = X_train.corr().abs()

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(
            correlation_matrix.shape
        ),
        k=1
    ).astype(bool)
)

high_correlation_features = [
    column
    for column in upper_triangle.columns
    if any(
        upper_triangle[column] > 0.95
    )
]

print(
    f"Highly correlated features: "
    f"{len(high_correlation_features)}"
)

print("\nFeatures identified:")
print(high_correlation_features)

Highly correlated features: 23

Features identified:
['Total_Backward_Packets', 'Total_Length_of_Bwd_Packets', 'Fwd_Packet_Length_Std', 'Bwd_Packet_Length_Mean', 'Bwd_Packet_Length_Std', 'Fwd_IAT_Total', 'Fwd_IAT_Max', 'Fwd_Packets_s', 'Packet_Length_Std', 'SYN_Flag_Count', 'CWE_Flag_Count', 'ECE_Flag_Count', 'Average_Packet_Size', 'Avg_Fwd_Segment_Size', 'Avg_Bwd_Segment_Size', 'Fwd_Header_Length.1', 'Subflow_Fwd_Packets', 'Subflow_Fwd_Bytes', 'Subflow_Bwd_Packets', 'Subflow_Bwd_Bytes', 'Idle_Mean', 'Idle_Max', 'Idle_Min']


In [9]:
# =====================================================
# REMOVE HIGHLY CORRELATED FEATURES
# =====================================================

X_train_reduced = X_train.drop(
    columns=high_correlation_features
)

X_test_reduced = X_test.drop(
    columns=high_correlation_features
)

print(
    f"Original features : "
    f"{X_train.shape[1]}"
)

print(
    f"Final features    : "
    f"{X_train_reduced.shape[1]}"
)

Original features : 70
Final features    : 47


In [10]:
# =====================================================
# VERIFY FEATURE CONSISTENCY
# =====================================================

train_features = set(
    X_train_reduced.columns
)

test_features = set(
    X_test_reduced.columns
)

print(
    "Same features in train and test:",
    train_features == test_features
)

print(
    "Training feature count:",
    len(train_features)
)

print(
    "Testing feature count:",
    len(test_features)
)

Same features in train and test: True
Training feature count: 47
Testing feature count: 47


In [11]:
# =====================================================
# FEATURE SELECTION SUMMARY
# =====================================================

feature_selection_summary = pd.DataFrame({
    "Metric": [
        "Original Features",
        "Constant Features Removed",
        "Highly Correlated Features Removed",
        "Final Features"
    ],
    "Value": [
        X.shape[1],
        len(constant_features),
        len(high_correlation_features),
        X_train_reduced.shape[1]
    ]
})

feature_selection_summary

,Metric,Value
0,Original Features,78
1,Constant Features Removed,8
2,Highly Correlated Features Removed,23
3,Final Features,47


In [12]:
# =====================================================
# SAVE FEATURE SELECTION INFORMATION
# =====================================================

selected_features = X_train_reduced.columns.tolist()

pd.DataFrame({
    "Selected_Feature": selected_features
}).to_csv(
    METRICS_PATH / "selected_features.csv",
    index=False
)

pd.DataFrame({
    "Removed_Feature": (
        constant_features
        + high_correlation_features
    )
}).to_csv(
    METRICS_PATH / "removed_features.csv",
    index=False
)

feature_selection_summary.to_csv(
    METRICS_PATH / "feature_selection_summary.csv",
    index=False
)

print("Feature selection information saved.")

Feature selection information saved.


In [13]:
# =====================================================
# SAVE TRAINING AND TESTING DATA
# =====================================================

train_df = X_train_reduced.copy()
train_df["Label"] = y_train.values

test_df = X_test_reduced.copy()
test_df["Label"] = y_test.values

train_output = (
    PROCESSED_DATA_PATH
    / "CICIDS2017_train.csv"
)

test_output = (
    PROCESSED_DATA_PATH
    / "CICIDS2017_test.csv"
)

train_df.to_csv(
    train_output,
    index=False
)

test_df.to_csv(
    test_output,
    index=False
)

print("Training and testing datasets saved.")

print(
    f"\nTraining shape: {train_df.shape}"
)

print(
    f"Testing shape : {test_df.shape}"
)

Training and testing datasets saved.

Training shape: (2016638, 48)
Testing shape : (504160, 48)


In [14]:
# =====================================================
# FINAL FEATURE ENGINEERING VERIFICATION
# =====================================================

print("=" * 60)
print("FEATURE ENGINEERING COMPLETED")
print("=" * 60)

print(
    f"Training rows    : "
    f"{train_df.shape[0]:,}"
)

print(
    f"Testing rows     : "
    f"{test_df.shape[0]:,}"
)

print(
    f"Final features   : "
    f"{train_df.shape[1] - 1}"
)

print(
    f"Training missing : "
    f"{train_df.isnull().sum().sum()}"
)

print(
    f"Testing missing  : "
    f"{test_df.isnull().sum().sum()}"
)

print(
    "\nTrain/Test features identical:",
    list(
        train_df.columns[:-1]
    ) == list(
        test_df.columns[:-1]
    )
)

FEATURE ENGINEERING COMPLETED
Training rows    : 2,016,638
Testing rows     : 504,160
Final features   : 47
Training missing : 0
Testing missing  : 0

Train/Test features identical: True
